<a href="https://colab.research.google.com/github/mahshidkhatiri/ml-from-scratch/blob/main/Linear_Regression_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NumPy + Linear Regression


**Goal:** build linear regression from scratch with NumPy, train it using gradient descent, and verify it with tests.

**Today’s finish line:** you can explain and implement the complete loop

$$X, y \rightarrow \hat y = Xw+b \rightarrow \text{MSE} \rightarrow \nabla_w, \nabla_b \rightarrow \text{updated } w,b$$

### Suggested timing (2.5 hours)
- 3:30–4:00 — NumPy warm-up
- 4:00–4:45 — predictions and loss
- 4:45–4:55 — break
- 4:55–5:30 — gradients and training
- 5:30–5:55 — tests and exam
- 5:55–6:00 — reflection and commit

> Work in order. Try each **YOUR TURN** cell before opening the matching answer near the end.

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)
print("NumPy version:", np.__version__)

NumPy version: 2.1.3


## 1. NumPy foundations

A dataset is stored as a matrix $X$. Each **row** is one observation and each **column** is one feature.

If $X$ has shape $(m,n)$ and $w$ has shape $(n,)$, then $Xw$ has shape $(m,)$. This gives one prediction per observation.

In [ ]:
# Worked example: 3 observations, 2 features
X_demo = np.array([[1., 2.],
                   [2., 0.],
                   [3., 1.]])
w_demo = np.array([2., -1.])
b_demo = 0.5

pred_demo = X_demo @ w_demo + b_demo
print("X shape:", X_demo.shape)
print("w shape:", w_demo.shape)
print("predictions:", pred_demo, "shape:", pred_demo.shape)

X shape: (3, 2)
w shape: (2,)
predictions: [0.5 4.5 5.5] shape: (3,)


### YOUR TURN 1 — shapes and matrix multiplication
Predict apartment prices using two features: area and number of bedrooms.

1. Before running the cell, write down the expected shape of `predictions`.
2. Replace `None` with one vectorized NumPy expression. Do not use a loop.

In [ ]:
X_apts = np.array([[700., 1.], [900., 2.], [1200., 3.]])
w_apts = np.array([0.25, 20.0])
b_apts = 30.0

predictions = X_apts@w_apts+b_apts  # TODO: Xw + b

# Uncomment after completing the TODO
assert predictions.shape == (3,)
np.testing.assert_allclose(predictions, [225., 295., 390.])
print("Correct! Predictions:", predictions)

Correct! Predictions: [225. 295. 390.]


## 2. The model and the loss

Linear regression predicts

$$\hat y_i = x_i^T w + b$$

The residual is $\hat y_i-y_i$. Mean squared error measures average squared residual size:

$$J(w,b)=\frac{1}{m}\sum_{i=1}^{m}(\hat y_i-y_i)^2$$

Squaring makes negative and positive errors count equally and penalizes large errors more strongly. MSE is a loss, not a distance norm: it averages **squared** values and does not take the square root.

In [ ]:
# Worked example
y_true_demo = np.array([3., 5., 7.])
y_pred_demo = np.array([2., 5., 9.])
residuals = y_pred_demo - y_true_demo
mse_demo = np.mean(residuals ** 2)
print("residuals:", residuals)
print("MSE:", mse_demo)  # (1 + 0 + 4) / 3

residuals: [-1.  0.  2.]
MSE: 1.6666666666666667


### YOUR TURN 2 — implement prediction and MSE
Complete both functions without loops. Keep outputs as NumPy arrays/scalars.

In [ ]:
def predict(X, w, b):
    """Return one prediction per row of X."""

    return X@w+b

def mse(y_true, y_pred):
    return np.mean((y_pred-y_true)**2)

# Uncomment after completing both functions
np.testing.assert_allclose(predict(X_demo, w_demo, b_demo), [0.5, 4.5, 5.5])
assert np.isclose(mse(np.array([1., 2.]), np.array([2., 4.])), 2.5)
print("Prediction and MSE tests passed.")

Prediction and MSE tests passed.


## 3. Gradient descent

A gradient tells us the direction in which the loss increases fastest. To reduce loss, move in the opposite direction. For MSE:

$$\nabla_w J = \frac{2}{m}X^T(\hat y-y), \qquad \frac{\partial J}{\partial b}=\frac{2}{m}\sum_{i=1}^m(\hat y_i-y_i)$$

Update rule with learning rate $\alpha$:

$$w \leftarrow w-\alpha\nabla_wJ, \qquad b \leftarrow b-\alpha\frac{\partial J}{\partial b}$$

### Shape check
- `error = y_pred - y`: $(m,)$
- `X.T`: $(n,m)$
- `X.T @ error`: $(n,)$ — exactly the shape of `w`
- `db`: scalar — exactly the shape of `b`

### YOUR TURN 3 — implement gradients
First answer on paper: why is `X.T` necessary in `dw`? Then complete the function.

In [ ]:
def gradients(X, y, w, b):
    """Return (dw, db) for the MSE objective."""
    m = X.shape[0]
    y_pred = predict(X, w, b)
    error = y_pred - y
    dw = (2/m)*X.T@(y_pred-y)
    db = (2/m)*np.sum(y_pred-y)
    return dw, db

# Uncomment after completing gradients
X_g = np.array([[1.], [2.]])
y_g = np.array([2., 4.])
dw_g, db_g = gradients(X_g, y_g, np.array([0.]), 0.0)
np.testing.assert_allclose(dw_g, [-10.])
assert np.isclose(db_g, -6.0)
print("Gradient test passed.")

Gradient test passed.


## 4. Train the model

We will use synthetic data so the true relationship is known. The noise makes the problem realistic: the fitted parameters should be close to, but not exactly, the true values.

In [ ]:
# Two features with different meanings but similar numeric scale
m, n = 120, 2
X = rng.normal(size=(m, n))
true_w = np.array([3.0, -2.0])
true_b = 1.5
y = X @ true_w + true_b + rng.normal(scale=0.25, size=m)

print("X:", X.shape, "y:", y.shape)
print("First observation:", X[0], "target:", round(y[0], 3))

X: (120, 2) y: (120,)
First observation: [ 0.3047 -1.04  ] target: 4.275


### YOUR TURN 4 — write the training loop

Algorithm:
1. Initialize $w$ and $b$ to zero.
2. Calculate gradients.
3. Update parameters in the negative-gradient direction.
4. Record the loss.
5. Repeat.

Be careful: use `w -= learning_rate * dw`, not `w += ...`.

In [ ]:
def fit_linear_regression(X, y, learning_rate=0.05, epochs=300):
    w = np.zeros(X.shape[1])
    b = 0.0
    history = []

    for _ in range(epochs):
        y_pred=X@w+b
        MSE=np.mean((y_pred-y)**2)
        dw=(2/m)*X.T@(y_pred-y)
        db=(2/m)*np.sum(y_pred-y)

        w-=learning_rate*dw
        b-=learning_rate*db
        history.append(MSE)
        # TODO: calculate dw and db
        # TODO: update w and b
        # TODO: append the current MSE to history


    return w, b, np.array(history)

# Uncomment after completing the function
learned_w, learned_b, history = fit_linear_regression(X, y)
print("True w:", true_w, "Learned w:", learned_w)
print("True b:", true_b, "Learned b:", round(learned_b, 4))
print("Initial/final loss:", history[0], history[-1])

True w: [ 3. -2.] Learned w: [ 3.005  -2.0201]
True b: 1.5 Learned b: 1.5162
Initial/final loss: 10.86878189861996 0.06503148559418218


## 5. Test your implementation

Tests are executable claims about expected behavior. Run these only after completing the four tasks. If one fails, read its message and inspect the smallest relevant function first.

In [ ]:
def run_project_tests():
    # Prediction values and shape
    X_t = np.array([[1., 2.], [3., 4.]])
    w_t = np.array([2., -1.])
    pred = predict(X_t, w_t, 0.5)
    assert pred.shape == (2,), f"Expected shape (2,), got {pred.shape}"
    np.testing.assert_allclose(pred, [0.5, 2.5])

    # MSE known example and perfect prediction
    assert np.isclose(mse(np.array([1., 2.]), np.array([2., 4.])), 2.5)
    assert np.isclose(mse(np.array([1., 2.]), np.array([1., 2.])), 0.0)

    # Gradient values and dimensions
    dw, db = gradients(np.array([[1.], [2.]]), np.array([2., 4.]),
                       np.array([0.]), 0.0)
    assert dw.shape == (1,), f"Expected dw shape (1,), got {dw.shape}"
    np.testing.assert_allclose(dw, [-10.])
    assert np.isclose(db, -6.0)

    # Training behavior
    w_fit, b_fit, losses = fit_linear_regression(X, y, 0.05, 300)
    assert len(losses) == 300
    print(losses[-1],losses[0])
    assert losses[-1] < losses[0] * 0.05, "Loss did not decrease enough"
    np.testing.assert_allclose(w_fit, true_w, atol=0.15)
    assert np.isclose(b_fit, true_b, atol=0.15)
    print("✅ All project tests passed.")

run_project_tests()

0.06503148559418218 10.86878189861996
✅ All project tests passed.


## 6. Final examination — no notes

Write your answers before checking the key.

**Q1 — Shapes and meaning**  
$X$ has shape `(100, 4)`, `w` has shape `(4,)`, and `b` is a scalar. What is the shape of `X @ w + b`? What does each output value represent?

**Q2 — Manual computation**  
For `X = [[1, 2], [3, 1]]`, `w = [2, -1]`, and `b = 0.5`, calculate both predictions manually. If `y = [1, 6]`, calculate the MSE.

**Q3 — Debugging**  
A student's loss increases rapidly and becomes `inf`. Give two plausible causes and one diagnostic action for each.

**Q4 — Gradient reasoning**  
Why does `dw` use `X.T @ error`? Explain both its shape and its meaning.

**Q5 — Coding challenge**  
Implement `r2_score` without scikit-learn using
$$R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}.$$

In [ ]:
def r2_score(y_true, y_pred):
    return 1-((np.sum((y_true-y_pred)**2))/(np.sum((y_true-np.mean(y_true))**2)))

# Self-check after implementing
assert np.isclose(r2_score(np.array([1., 2., 3.]), np.array([1., 2., 3.])), 1.0)
assert np.isclose(r2_score(np.array([1., 2., 3.]), np.array([2., 2., 2.])), 0.0)

## 7. Hints — open only when stuck

<details><summary>Hint for prediction and MSE</summary>Prediction needs <code>@</code>. For MSE, subtract, square elementwise with <code>** 2</code>, and average.</details>

<details><summary>Hint for gradients</summary>Let <code>error = y_pred - y</code>. Multiply <code>X.T @ error</code> by <code>2 / m</code>. The bias gradient is <code>2 * np.mean(error)</code>.</details>

<details><summary>Hint for training</summary>Call <code>gradients</code>, subtract learning rate times each gradient, then calculate MSE using the updated parameters.</details>

<details><summary>Hint for R²</summary>Calculate a residual sum of squares and a total sum of squares, then use the formula above.</details>

## 8. Answer key — check only after attempting

**Q1.** Shape `(100,)`; each value is the predicted target for one observation.

**Q2.** Predictions: $0.5$ and $5.5$. Residuals relative to `[1, 6]` are `[-0.5, -0.5]`, so MSE is $0.25$.

**Q3.** Examples: (1) learning rate too large — inspect loss over the first few iterations and reduce the rate; (2) features on very different or very large scales — inspect column ranges and standardize them. A sign error in the update is another common cause.

**Q4.** `X.T` changes the shape from `(m,n)` to `(n,m)`, so `(n,m) @ (m,)` produces `(n,)`, matching `w`. Conceptually, it combines every observation's error with each feature value, measuring how each weight contributed to the errors.

In [ ]:
# Reference solutions — compare with yours; do not paste blindly
def predict_solution(X, w, b):
    return X @ w + b

def mse_solution(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)

def gradients_solution(X, y, w, b):
    error = predict_solution(X, w, b) - y
    m = X.shape[0]
    return (2 / m) * (X.T @ error), 2 * np.mean(error)

def fit_solution(X, y, learning_rate=0.05, epochs=300):
    w, b = np.zeros(X.shape[1]), 0.0
    history = []
    for _ in range(epochs):
        dw, db = gradients_solution(X, y, w, b)
        w -= learning_rate * dw
        b -= learning_rate * db
        history.append(mse_solution(y, predict_solution(X, w, b)))
    return w, b, np.array(history)

def r2_score_solution(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

In [ ]:
# Validate the reference implementation
w_ref, b_ref, loss_ref = fit_solution(X, y)
assert loss_ref[-1] < loss_ref[0] * 0.05
np.testing.assert_allclose(w_ref, true_w, atol=0.15)
assert np.isclose(b_ref, true_b, atol=0.15)
print("Reference implementation verified. Final loss:", round(loss_ref[-1], 5))

Reference implementation verified. Final loss: 0.06503


## 9. Reflection and project wrap-up

Write 1–2 sentences for each:

1. I can now explain...
2. The bug or idea that challenged me most was...
3. I know my model learned because...
4. Tomorrow, I would improve this project by...

### Completion checklist
- [ ] I completed all four YOUR TURN tasks without copying first.
- [ ] All project tests pass.
- [ ] I answered all five exam questions.
- [ ] I compared my answers with the key.
- [ ] I can explain the shapes in $X^T(\hat y-y)$.
- [ ] I saved the notebook and committed it to Git.